# 01 — A first look at Online Retail

## Goal
Learn how to load a real dataset and understand what one row represents before calculating sales metrics. This is the first lesson in our E-commerce Sales & Customer Analytics project.

**In this lesson:** load the workbook, preview five rows, inspect types and coverage, and flag questions for cleaning. No source rows are changed or removed.

Source: Chen, D. (2015). [Online Retail, UCI](https://doi.org/10.24432/C5BW33), CC BY 4.0. Prices are in GBP. This is historical data, not a live store feed.

## Setup
Launch JupyterLab using the project's `.venv` as described in `README.md`. Select the **E-commerce Analytics (.venv)** kernel if prompted. Run the cells from top to bottom; loading the Excel workbook may take a minute.

`pandas` gives us a **DataFrame**: a table with named columns and numbered rows. We use `Path` to locate files without hardcoding a machine-specific path.

In [1]:
from pathlib import Path
from hashlib import sha256
import json
import sys
import pandas as pd
from IPython.display import display

# Support launching from either the project root or the notebooks folder.
current_folder = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in [current_folder, *current_folder.parents]
     if (folder / 'scripts' / 'download_data.py').is_file()
     and (folder / 'requirements.txt').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from within the project folder.')

DATA_FILE = PROJECT_ROOT / 'data' / 'raw' / 'Online Retail.xlsx'
pd.set_option('display.max_columns', 8)
print(f'Python: {sys.version.split()[0]} | pandas: {pd.__version__}')
print(f'Project folder: {PROJECT_ROOT}')

Python: 3.14.6 | pandas: 3.1.1
Project folder: C:\Users\Dev\Documents\ChatGPT\E-commerce Sales & Customer Analytics


## Steps
### 1. Verify and load the source
The downloader records a checksum, which acts as a fingerprint of the source file. We verify it before loading. IDs are labels, so we read them as text. `openpyxl` is the engine that reads this Excel format.

The eight columns come from the workbook itself. We inspect all raw rows; no customer filter or cleaning is applied.

In [2]:
if not DATA_FILE.exists():
    raise FileNotFoundError('Run scripts/download_data.py from the project folder first.')

manifest_file = DATA_FILE.parent / 'source_manifest.json'
manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
assert sha256(DATA_FILE.read_bytes()).hexdigest() == manifest['sha256'], 'Source file changed.'

retail_raw = pd.read_excel(
    DATA_FILE,
    sheet_name='Online Retail',
    engine='openpyxl',
    dtype={'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'string'},
)
print(f'Loaded {len(retail_raw):,} rows and {len(retail_raw.columns)} columns.')

Loaded 541,909 rows and 8 columns.


### 2. Preview five rows
`head()` shows the first five rows. Look for repeated invoice numbers: each row describes an item on an invoice, and one invoice can include several items.

**A row is not an order.** We will count distinct eligible purchase invoices when calculating order metrics later.

In [3]:
retail_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


### 3. Understand the columns
The data dictionary explains what each field means. This table checks the types pandas actually loaded. A non-null value is a value that is present; missing identifiers need special care in customer analysis.

In [4]:
column_summary = pd.DataFrame({
    'Data type': retail_raw.dtypes.astype(str),
    'Present rows': retail_raw.notna().sum(),
    'Missing rows': retail_raw.isna().sum(),
})
display(column_summary)

,Data type,Present rows,Missing rows
InvoiceNo,string,541909,0
StockCode,string,541909,0
Description,object,540455,1454
Quantity,int64,541909,0
InvoiceDate,datetime64[us],541909,0
UnitPrice,float64,541909,0
CustomerID,string,406829,135080
Country,str,541909,0


### 4. Inspect coverage and identifiers
These counts describe the raw file, including cancellations and adjustments. Distinct invoice IDs here should not be interpreted as completed purchase orders. Customer counts omit missing IDs.

The first and last timestamps describe the observed period. The final calendar month is incomplete, which will matter when comparing monthly sales.

In [5]:
raw_overview = pd.Series({
    'Invoice line items': len(retail_raw),
    'Distinct raw invoice IDs': retail_raw['InvoiceNo'].nunique(),
    'Distinct stock codes': retail_raw['StockCode'].nunique(),
    'Distinct known customer IDs': retail_raw['CustomerID'].nunique(),
    'First recorded timestamp': retail_raw['InvoiceDate'].min(),
    'Last recorded timestamp': retail_raw['InvoiceDate'].max(),
}, name='Raw file overview')
display(raw_overview.to_frame())

,Raw file overview
Invoice line items,541909
Distinct raw invoice IDs,25900
Distinct stock codes,4070
Distinct known customer IDs,4372
First recorded timestamp,2010-12-01 08:26:00
Last recorded timestamp,2011-12-09 12:50:00


## Checks
### 5. Confirm that we loaded the intended dataset
These checks compare the workbook shape and column names with UCI's description, and verify that invoice dates were parsed as dates. A failed check is a reason to investigate the input, not to edit the raw file to make the check pass.

In [6]:
expected_columns = [
    'InvoiceNo', 'StockCode', 'Description', 'Quantity',
    'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country',
]
assert retail_raw.columns.tolist() == expected_columns, 'Unexpected columns.'
assert retail_raw.shape == (541_909, 8), 'Unexpected dataset size.'
assert pd.api.types.is_datetime64_any_dtype(retail_raw['InvoiceDate'])
print('Source checksum, dataset shape, columns, and date type checks passed.')

Source checksum, dataset shape, columns, and date type checks passed.


## Next Steps
Before we calculate revenue or rank customers, we will inspect:

- Missing customer IDs: how can we keep anonymous sales without treating them as one customer?
- Invoice IDs starting with `C`: how should cancellations/credits enter revenue and order counts?
- Unusual quantities and prices: which rows are purchases, adjustments, or something else?
- Identical-looking rows: are they duplicates or repeated entries that need evidence before removal?

**Your checkpoint:** in the five-row preview, find two rows with the same `InvoiceNo`. Which columns change? Explain why counting rows would overstate the number of orders.

This notebook deliberately stops before cleaning. We will document the population and metric rules in `docs/analysis_decisions.md` as we investigate.